# Voice AI — Colab GPU sürümü

Bu notebook, kendi ses örneğinizle **İngilizce, Almanca, İspanyolca, İtalyanca ve Portekizce** konuşma oluşturmak için Voice AI arayüzünü başlatır. Ücretli API anahtarı gerekmez.

Başlamadan önce **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin. Bağlantıyı başkalarıyla paylaşmayın.

## 1. Model lisansı ve kullanım izni

XTTS-v2 model ağırlıkları [Coqui Public Model License](https://coqui.ai/cpml) kapsamındadır. Ayrıca yalnızca size ait veya açık kullanım izni bulunan sesleri yükleyin.

In [ ]:
accept_xtts_license = False #@param {type:"boolean"}
confirm_voice_permission = False #@param {type:"boolean"}

if not accept_xtts_license:
    raise ValueError("Devam etmek için XTTS-v2 model lisansını okuyup onaylayın.")
if not confirm_voice_permission:
    raise ValueError("Yalnızca size ait veya izinli ses kullanacağınızı onaylayın.")
print("✓ Onaylar kaydedildi.")

## 2. Kurulum

Büyük paketler ve model yalnızca Colab sunucusuna indirilir. İlk kurulum birkaç dakika sürebilir.

In [ ]:
import os, pathlib, subprocess, sys

repo = pathlib.Path("/content/Voice-AI")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Dpehect/Voice-AI.git", str(repo)], check=True)

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "backend/requirements.txt")], check=True)
subprocess.run(["npm", "ci", "--prefix", str(repo / "frontend")], check=True)

cloudflared = pathlib.Path("/content/cloudflared")
if not cloudflared.exists():
    subprocess.run(["wget", "-q", "-O", str(cloudflared), "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    cloudflared.chmod(0o755)

print("✓ Kurulum tamamlandı.")

## 3. Uygulamayı başlat

Bu hücre çalışırken oturumu açık tutun. İlk ses üretiminde model belleğe yükleneceği için bekleme normaldir.

In [ ]:
import os, pathlib, re, subprocess, sys, time, requests
from IPython.display import display, HTML

repo = pathlib.Path("/content/Voice-AI")
cloudflared = "/content/cloudflared"
processes = []

def wait_http(url, timeout=90):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            if requests.get(url, timeout=2).status_code < 500:
                return
        except requests.RequestException:
            pass
        time.sleep(1)
    raise RuntimeError(f"Servis başlatılamadı: {url}")

def tunnel(port):
    process = subprocess.Popen(
        [cloudflared, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    processes.append(process)
    deadline = time.time() + 60
    while time.time() < deadline:
        line = process.stdout.readline()
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if match:
            return match.group(0)
        if process.poll() is not None:
            break
    raise RuntimeError("Geçici bağlantı oluşturulamadı. Hücreyi yeniden çalıştırın.")

backend_env = os.environ.copy()
backend_env.update({"COQUI_TOS_AGREED": "1", "VOICE_AI_CORS_ORIGINS": "*"})
backend = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=repo / "backend", env=backend_env
)
processes.append(backend)
wait_http("http://127.0.0.1:8000/api/health")
api_url = tunnel(8000)

frontend_env = os.environ.copy()
frontend_env["NEXT_PUBLIC_API_URL"] = api_url
subprocess.run(["npm", "run", "build"], cwd=repo / "frontend", env=frontend_env, check=True)
frontend = subprocess.Popen(
    ["npm", "run", "start", "--", "--port", "3000"],
    cwd=repo / "frontend", env=frontend_env
)
processes.append(frontend)
wait_http("http://127.0.0.1:3000")
app_url = tunnel(3000)

display(HTML(f'''
<div style="font-family:system-ui;padding:24px;border:1px solid #dfe3dc;border-radius:16px;background:#f9fff0">
  <h2 style="margin:0 0 8px">✓ Voice AI hazır</h2>
  <p>Uygulamayı açmak için aşağıdaki düğmeye basın. Bu bağlantıyı paylaşmayın.</p>
  <a href="{app_url}" target="_blank" style="display:inline-block;padding:12px 18px;border-radius:10px;background:#253c31;color:#fff;text-decoration:none;font-weight:700">Voice AI'ı aç</a>
</div>
'''))